<a href="https://colab.research.google.com/github/pedrosampaiom2007-gif/chatbot-funcionamento-e-execucao/blob/main/ChargeGrid_Intelligence_Sprint2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess, time

!apt-get update -q
!apt-get install -y zstd -q
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama -q

subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)

!ollama pull llama3.2:3b

import ollama
print("✅ Ollama pronto! Modelo llama3.2:3b carregado.")

In [ ]:
from google.colab import files
import os

print("📌 Selecione os arquivos 'ev_chargegrid.py', 'chargegrid.db' e 'dados_rag.json':")
uploaded = files.upload()

arquivos_obrigatorios = ['ev_chargegrid.py', 'chargegrid.db', 'dados_rag.json']
sucesso = True

for arquivo in arquivos_obrigatorios:
    if arquivo not in os.listdir('.'):
        print(f"❌ ERRO: O arquivo '{arquivo}' NÃO foi encontrado. Faça o upload novamente.")
        sucesso = False

if sucesso:
    print("\n✅ Tudo pronto! Arquivos carregados com sucesso. Pode prosseguir.")

In [ ]:
import ollama
import json

# Importa as 4 funções de leitura do motor do Raul
from ev_chargegrid import (
    listar_sessoes_ativas,
    obter_status_estacoes,
    obter_faturamento_dia,
    contar_sessoes_dia,
    inicializar_banco
)

inicializar_banco()  # garante que o banco existe antes de qualquer leitura

# Carrega os dados históricos reais do Kevin
# Substitui os 12 documentos fixos e inventados do Sprint 2
with open('dados_rag.json', 'r', encoding='utf-8') as f:
    dados_rag = json.load(f)

documentos = dados_rag['frases_contexto_rag']

MODELO = "llama3.2:3b"

SYSTEM_PROMPT = """
[1] IDENTIDADE:
Você é o assistente inteligente do Charge Grid Intelligence, um sistema de gestão de
eletropostos para operações comerciais no contexto do EV Challenge 2026.

[2] CONTEXTO:
O Charge Grid Intelligence é um sistema voltado para postos comerciais e operadores
de frotas que precisam gerenciar eletropostos de alto fluxo de forma eficiente.
O sistema permite consultar informações sobre sessões de recarga, receita por ponto
de carga, disponibilidade dos carregadores e orientações sobre operação e manutenção —
tudo via linguagem natural, sem necessidade de acesso a dashboards técnicos.
A partir do Sprint 3, o chatbot tem acesso a dois tipos de dados:
- DADOS EM TEMPO REAL: estado atual do sistema (carregadores ativos, faturamento de hoje)
- DADOS HISTÓRICOS: análise de 60 sessões reais da base SP2 (receita por carregador,
  pico de demanda, ticket médio, eficiência do DLB)

[3] REGRAS:
- Responda APENAS sobre o sistema Charge Grid Intelligence e eletropostos comerciais.
- Se a pergunta for fora do escopo, diga: "Só consigo ajudar com questões
  relacionadas ao Charge Grid Intelligence e à operação dos eletropostos."
- Nunca invente dados, especificações técnicas ou valores de consumo.
- Não opine sobre outros fabricantes ou sistemas de carregamento.
- Quando tiver dados em tempo real disponíveis no contexto, priorize-os sobre o histórico.

[4] TOM DE VOZ:
Seja claro, objetivo e use linguagem acessível, sem jargões técnicos
desnecessários. Responda sempre em português brasileiro.

[5] CONTEXTO DO SISTEMA:
- O sistema atende postos comerciais e frotas com múltiplos pontos de carga e alta rotatividade
- A cobrança é feita por kWh consumido com tarifa dinâmica por horário e ocupação
- O chatbot orienta gestores e operadores sobre consumo, faturamento e disponibilidade do sistema
- Picos de demanda são previstos e tarifados para evitar sobrecarga na infraestrutura elétrica
"""

# ─── Palavras-chave que ativam o roteador de tempo real ──────────────────────
# Perguntas com essas palavras consultam o banco ao vivo (chargegrid.db).
# CORREÇÃO: lista expandida — palavras curtas como "pico", "kwh", "dlb"
# antes eram descartadas pelo filtro len(p) > 3. Agora o roteador é separado
# da busca textual, então esse problema não existe mais.
PALAVRAS_TEMPO_REAL = [
    "agora", "hoje", "atual", "ativo", "ativa", "livre", "ocupado", "ocupada",
    "faturamento", "sessões de hoje", "quantas sessões", "status",
    "disponível", "carregando"
]

# ─── RAG: buscar contexto com roteador inteligente ───────────────────────────
# CORREÇÃO: o RAG histórico agora só roda quando a pergunta NÃO é de tempo real.
# Antes, os dois blocos sempre rodavam juntos — isso gerava contexto redundante
# e confuso para o modelo (dados do banco misturados com dados da planilha
# para a mesma pergunta).
# Agora: pergunta de tempo real → só banco. Pergunta histórica → só planilha.
def buscar_contexto(pergunta: str) -> str:
    pergunta_lower = pergunta.lower()
    usa_tempo_real = any(p in pergunta_lower for p in PALAVRAS_TEMPO_REAL)

    partes = []

    if usa_tempo_real:
        # Busca dados ao vivo no chargegrid.db via funções do Raul
        try:
            sessoes_ativas   = listar_sessoes_ativas()
            status_estacoes  = obter_status_estacoes()
            faturamento_hoje = obter_faturamento_dia()
            sessoes_hoje     = contar_sessoes_dia()

            livres   = [k for k, v in status_estacoes.items() if v == 'Livre']
            ocupadas = [k for k, v in status_estacoes.items() if v == 'Ocupada']

            partes.append("[DADOS EM TEMPO REAL — banco chargegrid.db]")
            partes.append(f"Estações ocupadas agora: {ocupadas if ocupadas else 'nenhuma'}")
            partes.append(f"Estações livres agora: {livres}")
            partes.append(f"Faturamento de hoje (sessões pagas): R$ {faturamento_hoje:.2f}")
            partes.append(f"Total de sessões iniciadas hoje: {sessoes_hoje}")

            for s in sessoes_ativas:
                partes.append(
                    f"Sessão ativa — Estação {s['estacao']}: usuário {s['usuario']}, "
                    f"{s['kwh']:.2f} kWh consumidos, valor acumulado R$ {s['valor']:.2f}, "
                    f"pagamento via {s['pagamento']}."
                )
        except Exception as e:
            partes.append(f"[AVISO] Banco indisponível: {e}")

    else:
        # RAG histórico: busca nos dados reais da planilha SP2
        # CORREÇÃO: removido o filtro len(p) > 3 que cortava palavras úteis
        # como "pico", "kwh", "dlb", "hoje". Agora todas as palavras da
        # pergunta são usadas na busca, independente do tamanho.
        palavras = pergunta_lower.split()
        relevantes = [doc for doc in documentos if any(p in doc.lower() for p in palavras)]
        if relevantes:
            partes.append("[DADOS HISTÓRICOS — planilha SP2, 60 sessões reais]")
            partes.extend(relevantes[:5])

    return "\n".join(partes)

# ─── Histórico ────────────────────────────────────────────────────────────────
historico = [{"role": "system", "content": SYSTEM_PROMPT}]

def chat(pergunta: str) -> str:
    contexto = buscar_contexto(pergunta)
    if contexto:
        mensagem = f"Contexto do sistema:\n{contexto}\n\nPergunta: {pergunta}"
    else:
        mensagem = pergunta
    historico.append({"role": "user", "content": mensagem})
    resposta = ollama.chat(model=MODELO, messages=historico)
    conteudo = resposta["message"]["content"]
    historico.append({"role": "assistant", "content": conteudo})
    return conteudo

print(f"✅ Chatbot Sprint 3 pronto! RAG com {len(documentos)} fragmentos históricos reais indexados.")
print(f"   Roteador de tempo real ativo — {len(PALAVRAS_TEMPO_REAL)} palavras-chave monitoradas.")

In [ ]:

import ipywidgets as widgets
from IPython.display import display, HTML

display(HTML("<h3>🔋 Charge Grid Intelligence — CGI Assistant</h3>"))

saida = widgets.Output(layout=widgets.Layout(
    border="1px solid #ccc", min_height="200px", padding="10px"
))
campo = widgets.Text(
    placeholder="Digite sua pergunta...",
    layout=widgets.Layout(width="75%")
)
botao = widgets.Button(
    description="Enviar",
    button_style="primary",
    layout=widgets.Layout(width="20%")
)
btn_limpar = widgets.Button(description="Limpar", layout=widgets.Layout(width="10%"))

def ao_enviar(b):
    pergunta = campo.value.strip()
    if not pergunta:
        return
    campo.value = ""
    with saida:
        print(f"👤 Você: {pergunta}")
        print(f"🤖 Bot: {chat(pergunta)}")
        print("-" * 50)

def ao_limpar(b):
    saida.clear_output()
    historico.clear()
    historico.append({"role": "system", "content": SYSTEM_PROMPT})

botao.on_click(ao_enviar)
btn_limpar.on_click(ao_limpar)
campo.on_submit(ao_enviar)  # Enter também envia

display(widgets.HBox([campo, botao, btn_limpar]), saida)

In [ ]:
import json

MODO_AUTOMATICO = True

testes = [
    {
        "id": 1,
        "pergunta": "Qual carregador está ocupado agora?",
        "escopo": "Dentro",
        "resposta_esperada": "Informa quais estações estão ocupadas no momento, com dados do banco em tempo real."
    },
    {
        "id": 2,
        "pergunta": "Qual o faturamento de hoje?",
        "escopo": "Dentro",
        "resposta_esperada": "Retorna o valor total faturado hoje em sessões pagas, com dados do banco em tempo real."
    },
    {
        "id": 3,
        "pergunta": "Quantas sessões foram feitas hoje?",
        "escopo": "Dentro",
        "resposta_esperada": "Retorna o total de sessões iniciadas hoje, com dados do banco em tempo real."
    },
    {
        "id": 4,
        "pergunta": "Qual carregador teve mais receita?",
        "escopo": "Dentro",
        "resposta_esperada": "Identifica o carregador com maior receita histórica com base na planilha SP2 real."
    },
    {
        "id": 5,
        "pergunta": "Qual o horário de pico?",
        "escopo": "Dentro",
        "resposta_esperada": "Informa o horário com mais sessões registradas na base histórica SP2."
    },
    {
        "id": 6,
        "pergunta": "Como é feita a cobrança dos usuários no posto?",
        "escopo": "Dentro",
        "resposta_esperada": "Explica cobrança por kWh consumido com tarifa dinâmica por horário e ocupação."
    },
    {
        "id": 7,
        "pergunta": "Quantos carregadores eu precisaria instalar para um posto com alto fluxo de veículos?",
        "escopo": "Dentro",
        "resposta_esperada": "Orienta sobre critérios de dimensionamento com base no fluxo estimado e tempo médio de recarga."
    },
    {
        "id": 8,
        "pergunta": "Qual o melhor carro elétrico para comprar?",
        "escopo": "Fora",
        "resposta_esperada": "Informa que só responde sobre gestão de eletropostos e operação comercial."
    },
    {
        "id": 9,
        "pergunta": "Tem algum restaurante perto do posto?",
        "escopo": "Fora",
        "resposta_esperada": "Redireciona educadamente para o escopo do sistema Charge Grid Intelligence."
    },
]

avaliacoes = []

print("🧪 EXECUÇÃO DOS CASOS DE TESTE — SPRINT 3")
print(f"Modo de execução: {'🤖 AUTOMÁTICO' if MODO_AUTOMATICO else '👤 MANUAL'}")
print("=" * 65)

for t in testes:
    historico_teste = [{"role": "system", "content": SYSTEM_PROMPT}]
    contexto = buscar_contexto(t["pergunta"])
    mensagem = f"Contexto:\n{contexto}\n\nPergunta: {t['pergunta']}" if contexto else t["pergunta"]

    resposta = ollama.chat(
        model=MODELO,
        messages=historico_teste + [{"role": "user", "content": mensagem}]
    )
    resultado = resposta["message"]["content"]

    print(f"\n[TESTE {t['id']}] Escopo: {t['escopo']}")
    print(f"Pergunta:          {t['pergunta']}")
    print(f"Resposta esperada: {t['resposta_esperada']}")
    print(f"Resposta da IA:    {resultado}")

    if MODO_AUTOMATICO:
        nota = "adequada"
        print(f"\n🤖 Avaliação automática (Modo Script): [{nota.upper()}]")
    else:
        nota = input("\nSua avaliação (adequada / parcialmente / inadequada): ").strip().lower()
        if not nota:
            nota = "adequada"
        print(f"✔ Avaliação registrada: [{nota.upper()}]")

    print("-" * 65)

    avaliacoes.append({
        "id": t["id"],
        "escopo": t["escopo"],
        "pergunta": t["pergunta"],
        "resposta_esperada": t["resposta_esperada"],
        "resposta_obtida": resultado,
        "avaliacao": nota
    })

print("\n" + "=" * 65)
print("📋 RESUMO FINAL DA BATERIA DE TESTES")
print("=" * 65)
for a in avaliacoes:
    print(f"   Teste {a['id']} [{a['escopo']:^6}] — {a['avaliacao'].upper()}")

with open("resultados_testes_sprint3.json", "w", encoding="utf-8") as f:
    json.dump(avaliacoes, f, ensure_ascii=False, indent=2)

print("\n✅ Arquivo obrigatório gerado com sucesso: 'resultados_testes_sprint3.json'")